# Positional Embeddings in Attention: From Sinusoidal to RoPE

Transformers are permutation-invariant — without positional information, self-attention treats "the cat sat on the mat" identically to "mat the on sat cat the." **Positional embeddings** fix this by injecting position-dependent signals into the computation.

This tutorial explores two approaches:

1. **Sinusoidal Positional Encoding** ([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)) — the original scheme from *Attention Is All You Need*, which **adds** a fixed vector to each token embedding.
2. **Rotary Position Embeddings (RoPE)** ([Su et al., 2021](https://arxiv.org/abs/2104.09864)) — the modern approach used in LLaMA, Mistral, and most current LLMs, which **rotates** query and key vectors so that the attention dot product naturally encodes relative position.

We then benchmark three implementations of multi-head attention with RoPE to see how kernel fusion pays off:

| Implementation | What it does |
|---|---|
| **Naive PyTorch** | Explicit loop over heads, full attention matrix materialized in HBM |
| **`torch.compile`** | Same code, but PyTorch's graph compiler auto-fuses operators |
| **Triton Fused Kernel** | Hand-written [Triton](https://triton-lang.org/) kernel that fuses RoPE + scaled dot-product attention + online softmax into a single GPU pass (FlashAttention-style) |

### Scope

We focus on optimizing the **per-head attention** computation. The MHA wrapper still loops over heads in Python; a production kernel would add batch and head dimensions to the launch grid. We keep `head_dim` small (32) so that a full tile fits in SRAM.

## Setup

Sequence length 2048, hidden dimension 4096, 128 heads (head dim = 32). All tensors are `bfloat16` on GPU.

In [1]:
import torch, math
import timeit

DEVICE = 'cuda:4'

SEQ_LEN = 2048
HIDDEN_DIM = 4096
NUM_HEADS = 128
HEAD_DIM = HIDDEN_DIM // NUM_HEADS

Random input $x$ of shape `(seq_len, hidden_dim)` and four projection matrices ($W_Q, W_K, W_V, W_O$):

In [2]:
x = torch.randn(SEQ_LEN, HIDDEN_DIM, device=DEVICE, dtype=torch.bfloat16)
Wq = torch.randn(HIDDEN_DIM, HIDDEN_DIM, device=DEVICE, dtype=torch.bfloat16)
Wk = torch.randn(HIDDEN_DIM, HIDDEN_DIM, device=DEVICE, dtype=torch.bfloat16)
Wv = torch.randn(HIDDEN_DIM, HIDDEN_DIM, device=DEVICE, dtype=torch.bfloat16)
Wo = torch.randn(HIDDEN_DIM, HIDDEN_DIM, device=DEVICE, dtype=torch.bfloat16)

## 1. Sinusoidal Positional Encoding

The original Transformer ([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)) **adds** a fixed vector to each token embedding. For position $m$ and dimension index $i$:

$$\text{PE}(m, 2i) = \sin\!\left(\frac{m}{10000^{2i/d}}\right), \qquad \text{PE}(m, 2i{+}1) = \cos\!\left(\frac{m}{10000^{2i/d}}\right)$$

Each dimension pair oscillates at a different frequency ($\theta_i = 10000^{-2i/d}$), so nearby positions have similar encodings while distant ones differ. The encoding requires **no learned parameters** — it is simply summed with the token embedding before entering the attention layers.

This is elegant, but has a limitation: the positional signal lives in the residual stream and can be diluted or overwritten by later layers. RoPE (Section 2) addresses this by embedding position directly into the attention dot product.

In [3]:
def sinusoidal_pos_enc(seq_len, embed_dim, base=10000.0):
    pos = torch.arange(seq_len).unsqueeze(1)
    omega = torch.exp(torch.arange(0, embed_dim, 2) * -math.log(base) / embed_dim)
    angles = pos * omega
    enc = torch.zeros(seq_len, embed_dim)
    enc[:, 0::2] = torch.sin(angles)
    enc[:, 1::2] = torch.cos(angles)
    return enc

pe = sinusoidal_pos_enc(seq_len=4, embed_dim=8)
print("Positional encoding (4 positions, 8 dims):")
print(pe.round(decimals=3))

print("\nOriginal embeddings + positional encoding:")
x_demo = torch.randn((4, 8))
print((x_demo + pe).round(decimals=3))

Positional encoding (4 positions, 8 dims):
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8410,  0.5400,  0.1000,  0.9950,  0.0100,  1.0000,  0.0010,  1.0000],
        [ 0.9090, -0.4160,  0.1990,  0.9800,  0.0200,  1.0000,  0.0020,  1.0000],
        [ 0.1410, -0.9900,  0.2960,  0.9550,  0.0300,  1.0000,  0.0030,  1.0000]])

Original embeddings + positional encoding:
tensor([[-0.5440,  1.8750,  0.5920,  0.7530, -1.3620,  1.3340, -1.0300,  0.6460],
        [-0.0600,  0.2690,  1.9360,  0.4770, -0.1830, -0.1240, -0.0960,  0.6690],
        [ 0.6940,  0.6950,  1.7280, -0.0790,  0.2060,  2.9340, -0.3930,  1.0510],
        [-3.0910, -1.3080,  0.9230, -0.9080,  0.5590,  1.3180, -0.1570,  1.3160]])


## 2. Rotary Position Embeddings (RoPE)

RoPE ([Su et al., 2021](https://arxiv.org/abs/2104.09864)) takes a fundamentally different approach: instead of *adding* a vector, it **rotates** the query and key vectors in 2D subspaces before the dot product.

Pair adjacent dimensions of a head vector into 2D coordinates $(x_0, x_1), (x_2, x_3), \ldots$ and rotate each pair by a position-dependent angle:

$$R_\theta(x, m) = \begin{pmatrix} x_0 \cos m\theta_0 - x_1 \sin m\theta_0 \\ x_0 \sin m\theta_0 + x_1 \cos m\theta_0 \\ \vdots \end{pmatrix}$$

where $m$ is the token position and $\theta_i = 10000^{-2i/d}$ are the same fixed frequencies as sinusoidal encoding. Because rotations compose, $\langle R_\theta(q, m),\, R_\theta(k, n) \rangle$ depends only on the relative distance $m - n$.

**Why RoPE over sinusoidal?**
- Position is encoded in the **attention scores** themselves, not the residual stream — it can't be washed out by later layers.
- Naturally captures **relative** position without explicit relative-position bias matrices.
- Easily extends to longer sequences than seen during training (via frequency scaling).

We precompute `cos` and `sin` tables of shape `(seq_len, head_dim // 2)` — one entry per dimension pair — and apply the rotation to Q and K after projection but before the dot product. V is left unrotated.

In [4]:
def precompute_rope(head_dim, seq_len, base=10000.0, device='cpu'):
    assert head_dim % 2 == 0
    freqs = torch.exp(torch.arange(0, head_dim, 2, device=device) * -math.log(base) / head_dim)
    angles = torch.arange(seq_len, device=device).unsqueeze(-1) * freqs
    return torch.cos(angles), torch.sin(angles)  # each (seq_len, head_dim // 2)

def apply_rope_interleaved(x, cos, sin):
    """Interleaved RoPE: rotate adjacent dimension pairs (0,1), (2,3), ..."""
    a = x[..., 0::2]
    b = x[..., 1::2]
    a_ = a * cos - b * sin  # [a', b'] = [a, b] @ [[cos, sin], [-sin, cos]]
    b_ = a * sin + b * cos
    return torch.stack((a_, b_), dim=-1).flatten(-2)

rope_cos, rope_sin = precompute_rope(HEAD_DIM, SEQ_LEN, device=DEVICE)
print(f"rope_cos: {rope_cos.shape}  rope_sin: {rope_sin.shape}")

q = (x @ Wq).view(SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(0, 1)
k = (x @ Wk).view(SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(0, 1)
print(f"q: {q.shape}  -> after RoPE: {apply_rope_interleaved(q, rope_cos, rope_sin).shape}")
print(f"k: {k.shape}  -> after RoPE: {apply_rope_interleaved(k, rope_cos, rope_sin).shape}")

rope_cos: torch.Size([2048, 16])  rope_sin: torch.Size([2048, 16])
q: torch.Size([128, 2048, 32])  -> after RoPE: torch.Size([128, 2048, 32])
k: torch.Size([128, 2048, 32])  -> after RoPE: torch.Size([128, 2048, 32])


## 3. Benchmarking Multi-Head Attention with RoPE

With positional embeddings covered, we now implement and benchmark the full attention computation. For a single head:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q_\text{rot}\, K_\text{rot}^T}{\sqrt{d_k}}\right) V$$

where $Q_\text{rot}$ and $K_\text{rot}$ have RoPE applied. We compare three implementations of this, each progressively more fused.

### 3.1 Naive PyTorch

Compute attention head-by-head in a Python loop. Simple and readable, but the full $(\text{seq\_len} \times \text{seq\_len})$ attention matrix is materialized in HBM and the loop prevents cross-head parallelism.

In [5]:
def attention_head(q, k, v, rope_cos, rope_sin):
    seq_len = q.shape[-2]
    head_dim = q.shape[-1]
    q_ = apply_rope_interleaved(q, rope_cos, rope_sin)
    k_ = apply_rope_interleaved(k, rope_cos, rope_sin)
    attn_scores = q_ @ torch.transpose(k_, -1, -2) / math.sqrt(head_dim) # (seq_len, seq_len)
    mask = torch.triu(torch.ones((seq_len, seq_len), device=q.device), diagonal=1).bool()
    attn_scores = torch.masked_fill(attn_scores, mask, float('-inf'))
    attn_output = torch.softmax(attn_scores, dim=-1) @ v # (seq_len, head_dim)
    return attn_output

def MHA_fwd(x, Wq, Wk, Wv, Wo, rope_cos, rope_sin, NUM_HEADS):
    # Wq, Wk, Wv are weight matrices of shape (hidden_dim, hidden_dim) 
    # q, k, v projections of x

    with torch.cuda.device(DEVICE), torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
        q = x @ Wq # (seq_len, hidden_dim)
        k = x @ Wk # (seq_len, hidden_dim)
        v = x @ Wv # (seq_len, hidden_dim)

        q = q.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
        k = k.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
        v = v.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)

        attn_output = torch.hstack([attention_head(q[i], k[i], v[i], rope_cos, rope_sin) for i in range(NUM_HEADS)])

        return attn_output @ Wo # (seq_len, hidden_dim)


Benchmark over 100 iterations (run a couple of times for stable numbers):

In [6]:
timeit.timeit(lambda: MHA_fwd(x, Wq, Wk, Wv, Wo, rope_cos, rope_sin, NUM_HEADS), number=100)

5.362242985982448

### 3.2 `torch.compile`

`torch.compile` traces the computation graph and applies automatic kernel fusion and operator rewriting. We compile just `attention_head` — same code, no manual optimization.

In [7]:
compiled_attention_head = torch.compile(attention_head)

def compiled_MHA_fwd(x, Wq, Wk, Wv, Wo, rope_cos, rope_sin, NUM_HEADS):
    # Wq, Wk, Wv are weight matrices of shape (hidden_dim, hidden_dim) 
    # q, k, v projections of x
    with torch.cuda.device(DEVICE), torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
        q = x @ Wq # (seq_len, hidden_dim)
        k = x @ Wk # (seq_len, hidden_dim)
        v = x @ Wv # (seq_len, hidden_dim)

        q = q.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
        k = k.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
        v = v.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)

        attn_output = torch.hstack([compiled_attention_head(q[i], k[i], v[i], rope_cos, rope_sin) for i in range(NUM_HEADS)])

        return attn_output @ Wo # (seq_len, hidden_dim)

In [8]:
timeit.timeit(lambda: compiled_MHA_fwd(x, Wq, Wk, Wv, Wo, rope_cos, rope_sin, NUM_HEADS), number=100)

2.669921403983608

### 3.3 Triton Fused Kernel

Now we bring everything — RoPE, scaled dot-product attention, and softmax — into a **single Triton kernel**. Instead of materializing the full $(\text{seq\_len} \times \text{seq\_len})$ attention matrix in HBM, we tile over the K/V sequence and accumulate the output using **online softmax** ([Milakov & Gimelshein, 2018](https://arxiv.org/abs/1805.02867)), the core idea behind [FlashAttention](https://arxiv.org/abs/2205.14135).

**Why is this faster?** The naive approach writes the full attention matrix to HBM and reads it back — $O(N^2)$ memory traffic. The fused kernel keeps intermediate results (scores, softmax denominators) in SRAM, reducing memory traffic to $O(N)$.

#### The kernel

Each Triton program handles one `BLOCK_SEQ`-sized tile of Q rows and iterates over all K/V tiles. RoPE is applied inline — no separate pass.

**Fusing interleaved RoPE.** We load the even columns (`q[..., 0::2]`) and odd columns (`q[..., 1::2]`) of Q and K via gather pointers, then apply the 2D rotation using the precomputed `cos`/`sin` tables. Since Triton can't interleave two half-dim tensors back into a full-dim tensor in registers, we exploit the identity:

$$Q_\text{rot}\, K_\text{rot}^T = Q_a' \cdot K_a'^T + Q_b' \cdot K_b'^T$$

Two half-dimension `tl.dot` calls replace one full-dimension dot, avoiding any need to reconstruct the interleaved layout.

**Kernel structure:**
1. Load Q's even/odd columns, load `cos`/`sin` for Q's positions, apply rotation → $Q_a', Q_b'$
2. Loop over K/V tiles of `BLOCK_SEQ` rows:
   - Load K's even/odd columns + `cos`/`sin` for K's positions → $K_a', K_b'$
   - Compute `scores = dot(Q_a', K_a'^T) + dot(Q_b', K_b'^T)`
   - Online softmax update, accumulate `P @ V`
3. Normalize and store output

In [9]:
import torch
import triton
import triton.language as tl


@triton.jit
def fused_attn_fwd_kernel(
    Q_ptr, K_ptr, V_ptr, O_ptr,
    COS_ptr, SIN_ptr,
    stride_q_row, stride_qd,
    stride_k_row, stride_kd,
    stride_v_row, stride_vd,
    stride_o_row, stride_od,
    stride_cos_row, stride_cosd,
    stride_sin_row, stride_sind,
    SEQ_LEN, HEAD_DIM,
    sm_scale,
    BLOCK_SEQ: tl.constexpr,
    BLOCK_HDIM: tl.constexpr,
):
    pid_m = tl.program_id(0)

    offs_q = pid_m * BLOCK_SEQ + tl.arange(0, BLOCK_SEQ)
    offs_kv = tl.arange(0, BLOCK_SEQ)
    offs_d = tl.arange(0, BLOCK_HDIM)
    offs_half_d = tl.arange(0, BLOCK_HDIM // 2)
    offs_a = tl.arange(0, BLOCK_HDIM // 2) * 2       # [0, 2, 4, 6, ...]
    offs_b = tl.arange(0, BLOCK_HDIM // 2) * 2 + 1   # [1, 3, 5, 7, ...]

    q_half_mask = (offs_q[:, None] < SEQ_LEN) & (offs_half_d[None, :] < HEAD_DIM // 2)

    # ---- Load Q even/odd columns and apply RoPE ----
    q_a_ptrs = Q_ptr + offs_q[:, None] * stride_q_row + offs_a[None, :] * stride_qd
    q_b_ptrs = Q_ptr + offs_q[:, None] * stride_q_row + offs_b[None, :] * stride_qd
    q_a = tl.load(q_a_ptrs, mask=q_half_mask, other=0.0)
    q_b = tl.load(q_b_ptrs, mask=q_half_mask, other=0.0)

    cos_q_ptrs = COS_ptr + offs_q[:, None] * stride_cos_row + offs_half_d[None, :] * stride_cosd
    sin_q_ptrs = SIN_ptr + offs_q[:, None] * stride_sin_row + offs_half_d[None, :] * stride_sind
    cos_q = tl.load(cos_q_ptrs, mask=q_half_mask, other=1.0)
    sin_q = tl.load(sin_q_ptrs, mask=q_half_mask, other=0.0)

    q_a_rot = q_a * cos_q - q_b * sin_q  # a' = a * cos - b * sin
    q_b_rot = q_a * sin_q + q_b * cos_q  # b' = a * sin + b * cos

    # ---- Online softmax accumulators ----
    m_i = tl.full((BLOCK_SEQ,), float("-inf"), tl.float32)
    l_i = tl.zeros((BLOCK_SEQ,), dtype=tl.float32)
    acc = tl.zeros((BLOCK_SEQ, BLOCK_HDIM), dtype=tl.float32)

    # ---- Loop over K/V tiles ----
    for start_m in range(0, SEQ_LEN, BLOCK_SEQ):
        k_idx = start_m + offs_kv
        k_half_mask = (k_idx[:, None] < SEQ_LEN) & (offs_half_d[None, :] < HEAD_DIM // 2)

        # Load K even/odd columns
        k_a_ptrs = K_ptr + k_idx[:, None] * stride_k_row + offs_a[None, :] * stride_kd
        k_b_ptrs = K_ptr + k_idx[:, None] * stride_k_row + offs_b[None, :] * stride_kd
        k_a = tl.load(k_a_ptrs, mask=k_half_mask, other=0.0)
        k_b = tl.load(k_b_ptrs, mask=k_half_mask, other=0.0)

        # K tiles cover different positions than Q — must load cos/sin for k_idx
        cos_k_ptrs = COS_ptr + k_idx[:, None] * stride_cos_row + offs_half_d[None, :] * stride_cosd
        sin_k_ptrs = SIN_ptr + k_idx[:, None] * stride_sin_row + offs_half_d[None, :] * stride_sind
        cos_k = tl.load(cos_k_ptrs, mask=k_half_mask, other=1.0)
        sin_k = tl.load(sin_k_ptrs, mask=k_half_mask, other=0.0)

        k_a_rot = k_a * cos_k - k_b * sin_k
        k_b_rot = k_a * sin_k + k_b * cos_k

        # Q_rot @ K_rot^T = Q_a' @ K_a'^T + Q_b' @ K_b'^T
        scores = (tl.dot(q_a_rot, tl.trans(k_a_rot)) + tl.dot(q_b_rot, tl.trans(k_b_rot))) * sm_scale
        causal_mask = k_idx[None, :] <= offs_q[:, None]
        valid_mask = (k_idx[None, :] < SEQ_LEN) & causal_mask
        scores = tl.where(valid_mask, scores, float("-inf"))

        # Online softmax update
        m_ij = tl.max(scores, axis=1)
        m_new = tl.maximum(m_i, m_ij)

        correction_factor = tl.exp(m_i - m_new)
        p = tl.exp(scores - m_new[:, None])
        l_new = l_i * correction_factor + tl.sum(p, axis=1)

        # V block (no RoPE on V)
        v_ptrs = V_ptr + k_idx[:, None] * stride_v_row + offs_d[None, :] * stride_vd
        v_mask = (k_idx[:, None] < SEQ_LEN) & (offs_d[None, :] < HEAD_DIM)
        v = tl.load(v_ptrs, mask=v_mask, other=0.0)

        acc = acc * correction_factor[:, None]
        acc = acc + tl.dot(p.to(v.dtype), v)

        m_i = m_new
        l_i = l_new

    out = acc / l_i[:, None]

    o_ptrs = O_ptr + offs_q[:, None] * stride_o_row + offs_d[None, :] * stride_od
    o_mask = (offs_q[:, None] < SEQ_LEN) & (offs_d[None, :] < HEAD_DIM)
    tl.store(o_ptrs, out, mask=o_mask)

#### Python wrapper

Allocates the output tensor, computes the launch grid, and passes the precomputed `cos`/`sin` tables (shape `(seq_len, head_dim // 2)`) directly to the kernel.

In [10]:
def fused_attention(q, k, v, rope_cos, rope_sin, sm_scale=None, BLOCK_SEQ=64):
    assert q.ndim == 2 and k.ndim == 2 and v.ndim == 2
    M, D = q.shape

    if sm_scale is None:
        sm_scale = 1.0 / (D ** 0.5)

    o = torch.empty((M, D), device=q.device, dtype=q.dtype)
    grid = (triton.cdiv(M, BLOCK_SEQ),)

    with torch.cuda.device(q.device):
        fused_attn_fwd_kernel[grid](
            q, k, v, o,
            rope_cos, rope_sin,
            q.stride(0), q.stride(1),
            k.stride(0), k.stride(1),
            v.stride(0), v.stride(1),
            o.stride(0), o.stride(1),
            rope_cos.stride(0), rope_cos.stride(1),
            rope_sin.stride(0), rope_sin.stride(1),
            M, D,
            sm_scale,
            BLOCK_SEQ=BLOCK_SEQ,
            BLOCK_HDIM=triton.next_power_of_2(D),
        )
    return o

#### Triton-powered MHA

Wire the fused kernel into the MHA function, replacing the per-head PyTorch attention:

In [11]:
def mha_fwd_triton(x, Wq, Wk, Wv, Wo, rope_cos, rope_nsin, NUM_HEADS):
    q = x @ Wq # (SEQ_LEN, HIDDEN_DIM)
    k = x @ Wk # (SEQ_LEN, HIDDEN_DIM)
    v = x @ Wv # (SEQ_LEN, HIDDEN_DIM)

    q = q.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
    k = k.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
    v = v.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)

    attn_output = torch.hstack([
        fused_attention(
            q[i], k[i], v[i], # (SEQ_LEN, HEAD_DIM)
            rope_cos, rope_nsin, # (SEQ_LEN, HEAD_DIM//2)
            sm_scale=None, BLOCK_SEQ=64,
        ) for i in range(NUM_HEADS)])

    return attn_output @ Wo

Benchmark the fused kernel against the earlier implementations:

In [12]:
rope_cos, rope_sin = precompute_rope(HEAD_DIM, SEQ_LEN, device=DEVICE)

In [13]:
timeit.timeit(lambda: mha_fwd_triton(x, Wq, Wk, Wv, Wo, rope_cos, rope_sin, NUM_HEADS), number=100)

1.9682543210219592

## Results

| Implementation | Time (100 iters) | Speedup |
|---|---|---|
| Naive PyTorch | ~5.36 s | 1.00x |
| `torch.compile` | ~2.66 s | ~2.01x |
| **Triton Fused (RoPE + attn + softmax)** | **~1.96 s** | **~2.73x** |

### Takeaways

- **Sinusoidal encoding** is simple and parameter-free but adds position to the residual stream where it can be diluted. **RoPE** embeds position directly in the attention dot product, which is why it dominates in modern LLMs.
- **Fusing RoPE into the attention kernel** avoids a separate pass over Q and K, saving memory bandwidth.
- **Online softmax** enables single-pass attention without materializing $O(N^2)$ intermediates — the core idea behind [FlashAttention](https://arxiv.org/abs/2205.14135).
- Even an untuned Triton kernel significantly outperforms naive PyTorch. For production, `torch.nn.functional.scaled_dot_product_attention` dispatches to optimized backends automatically.